# 📓 Day 6 Industry Project: Professional Email Writer

In this industry project, we build a utility to draft context-specific B2B outbound emails using **Role Prompting**, **Few-Shot examples**, and **Chain of Thought reasoning**.

---

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### 1. Structured Output Template with CoT planning

We prompt the model to reason about the prospect's pain point before drafting the cold email, ensuring the message is highly tailored.

In [ ]:
system_role = """
You are a world-class Sales Development Representative (SDR). Your emails are short, personalized, and value-focused.
You avoid buzzwords like 'synergy', 'game-changing', or 'next-gen'.

You must write in this format:
REASONING:
- Analyze the prospect's industry and pain points.
- Formulate a direct value proposition mapping to their needs.

EMAIL:
Subject: [Clear, short subject]
Hi [Name],

[Custom first line connecting to their role]
[Value proposition mapping to their pain point]
[Call to action (CTA) proposing a brief chat]

Best,
[Your Name]
"""

few_shot_data = """
Example 1:
PROSPECT DETAILS:
Name: John Doe
Role: Head of Payments
Company: RetailCorp
Pain Point: Credit card transaction fees are too high during peak holidays.

Output:
REASONING:
- The prospect is concerned with credit card transaction costs eating margins.
- Stripe offers custom pricing rates and high-volume discounts.
- Hook should mention holiday scaling.

EMAIL:
Subject: Custom rates for RetailCorp
Hi John,

Managing transaction costs during peak holiday traffic is a major headache for retail payment heads.

At Stripe, we help companies like RetailCorp reduce credit card transaction overhead by up to 15% through high-volume optimization.

Do you have 5 minutes this Thursday for a quick audit of your current payment processors?

Best,
Sarah
"""

### 2. Formulating and Running the Generator

In [ ]:
def generate_email(prospect_name: str, prospect_role: str, company: str, pain_point: str) -> str:
    user_content = f"""
    {few_shot_data}
    
    PROSPECT DETAILS:
    Name: {prospect_name}
    Role: {prospect_role}
    Company: {company}
    Pain Point: {pain_point}
    
    Output:
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_role},
            {"role": "user", "content": user_content}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

name = "David Miller"
role = "VP of Engineering"
company = "CloudScale"
pain = "Our cloud hosting costs are rising exponentially but we have idle servers."

output = generate_email(name, role, company, pain)
print(output)